In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from nba_roi.config import ROI_FILE, FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_excel(ROI_FILE)

# Rename columns for easier use (optional but helpful)
df = df.rename(columns={
    "Salary ($)": "Salary",
    "GAMES_PLAYED": "Games_Played",
    "ROI (Total_CF / Salary)": "ROI"
})

df.head()

In [ ]:
filtered_df = df[df["Games_Played"] >= 65]

top_10 = filtered_df.nlargest(10, "ROI")
bottom_10 = filtered_df.nsmallest(10, "ROI")

combined = pd.concat([top_10, bottom_10])

plt.figure(figsize=(12, 6))

# Color: green = good ROI, red = bad ROI
colors = ["green" if roi > 0 else "red" for roi in combined["ROI"]]

plt.barh(combined["Player"], combined["ROI"], color=colors)

plt.xlabel("ROI")
plt.title("Top 10 and Bottom 10 ROI Players (Min 65 Games)")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "quick_top_bottom_roi.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
team_avg = df.groupby("Team")["ROI"].mean().sort_values()

plt.figure(figsize=(12, 6))

plt.barh(team_avg.index, team_avg.values, color="skyblue")

plt.xlabel("Average ROI")
plt.title("Average Player ROI by Team")
plt.axvline(x=1, linestyle='--', linewidth=3, color='black')

plt.tight_layout()
plt.savefig(FIGURES_DIR / "quick_team_avg_roi.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create simple contract tiers based on salary
def classify_contract(salary):
    if salary < 5_000_000:
        return "Minimum"
    elif salary < 15_000_000:
        return "Mid-Level"
    elif salary < 35_000_000:
        return "Starter"
    else:
        return "Supermax"

df["Contract_Type"] = df["Salary"].apply(classify_contract)

# Color mapping
color_map = {
    "Minimum": "blue",
    "Mid-Level": "orange",
    "Starter": "green",
    "Supermax": "red"
}

colors = df["Contract_Type"].map(color_map)

plt.figure(figsize=(10, 6))

plt.scatter(df["Salary"] / 1_000_000, df["ROI"], c=colors)

plt.xlabel("Salary (Millions)")
plt.ylabel("ROI")
plt.title("ROI vs Salary by Contract Type")
plt.axhline(y=1, linestyle='--', linewidth=3, color='black')

# Legend
for contract, color in color_map.items():
    plt.scatter([], [], c=color, label=contract)

plt.legend(title="Contract Type")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "quick_roi_vs_salary.png", dpi=300, bbox_inches='tight')
plt.show()